# 07 Job Storage and IT Market Statistics

This notebook stores structured job posting data and creates basic IT market statistics.

Main goals:
- load structured job data extracted from a job posting
- store structured job data in a local SQLite database
- load cleaned IT job postings dataset
- analyze most common IT skills, tools and technologies
- analyze job categories, seniority levels and work modes
- analyze basic trends over time if date information is available
- save market statistics for later reporting

This notebook does not perform CV-job matching.

## 1. Imports and Settings

In [1]:
import json
import sqlite3
import pandas as pd
import numpy as np
import re

from pathlib import Path
from datetime import datetime

In [2]:
PROCESSED_JOBS_DIR = Path("../data/processed/jobs")
JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
DATABASE_DIR = Path("../database")
MARKET_STATS_DIR = Path("../outputs/market_statistics")

In [3]:
cleaned_jobs_path = PROCESSED_JOBS_DIR / "cleaned_it_job_postings.csv"
structured_job_path = JOB_EXTRACTION_DIR / "structured_job.json"
database_path = DATABASE_DIR / "jobs.sqlite"

## 2. Load Data

In [4]:
with open(structured_job_path, "r", encoding="utf-8") as file:
    structured_job = json.load(file)

In [5]:
structured_job_preview = {
    "job_title": structured_job.get("job_title"),
    "company_name": structured_job.get("company_name"),
    "location": structured_job.get("location"),
    "work_mode": structured_job.get("work_mode"),
    "employment_type": structured_job.get("employment_type"),
    "job_category": structured_job.get("job_category"),
    "required_skills": structured_job.get("required_skills", []),
    "nice_to_have_skills": structured_job.get("nice_to_have_skills", [])
}

In [6]:
structured_job_preview

{'job_title': 'Freelance Full-Stack Software Engineer',
 'company_name': 'SkillFit',
 'location': 'Remote',
 'work_mode': 'Remote',
 'employment_type': 'Freelance',
 'job_category': 'Software Development',
 'required_skills': ['JavaScript',
  'TypeScript',
  'React',
  'Node.js',
  'Python',
  'Express.js',
  'PostgreSQL',
  'MongoDB',
  'HTML5',
  'CSS3',
  'AWS',
  'Azure',
  'Google Cloud Platform',
  'version control',
  'testing',
  'CI/CD'],
 'nice_to_have_skills': ['none']}

In [7]:
df_jobs = pd.read_csv(cleaned_jobs_path)

In [8]:
df_jobs.head()

,title,description,company_name,formatted_work_type,work_type,formatted_experience_level,listed_time,original_listed_time,expiry,job_posting_url,clean_job_title,clean_job_description,combined_text,title_has_it_keyword,tech_keyword_count,it_job_category
0,Software Engineer,Job Description:GOYT is seeking a skilled and ...,GOYT,Part-time,PART_TIME,Unknown,1.713281e+12,1.713281e+12,1.715873e+12,https://www.linkedin.com/jobs/view/175485704/?...,Software Engineer,Job Description:GOYT is seeking a skilled and ...,software engineer job description:goyt is seek...,True,6,Frontend Development
1,Full Stack Engineer,"Location: Remote\nCompany Overview:SkillFit, a...",Ideando Inc,Full-time,FULL_TIME,Unknown,1.713493e+12,1.713493e+12,1.716085e+12,https://www.linkedin.com/jobs/view/2234533717/...,Full Stack Engineer,"Location: Remote Company Overview:SkillFit, a ...",full stack engineer location: remote company o...,False,17,Frontend Development
2,Salesforce Vlocity Developer,Role: Salesforce Vlocity DeveloperLocation: Ne...,SysMind,Contract,CONTRACT,Unknown,1.713211e+12,1.713211e+12,1.715803e+12,https://www.linkedin.com/jobs/view/3169712432/...,Salesforce Vlocity Developer,Role: Salesforce Vlocity DeveloperLocation: Ne...,salesforce vlocity developer role: salesforce ...,False,7,DevOps / Cloud
3,Data Architect,Request: Data ArchitectLocation: San Francisco...,Saxon AI,Contract,CONTRACT,Unknown,1.713537e+12,1.713537e+12,1.716129e+12,https://www.linkedin.com/jobs/view/3245063922/...,Data Architect,Request: Data ArchitectLocation: San Francisco...,data architect request: data architectlocation...,False,6,Backend Development
4,Senior Developer,This individual will work with a high performa...,USLI,Full-time,FULL_TIME,Associate,1.713538e+12,1.713538e+12,1.716130e+12,https://www.linkedin.com/jobs/view/3475933396/...,Senior Developer,This individual will work with a high performa...,senior developer this individual will work wit...,False,9,Frontend Development


## 3. Prepare Structured Job Data for SQLite

In [9]:
def list_to_json_string(value):
    if value is None:
        return json.dumps([], ensure_ascii=False)
    
    if isinstance(value, list):
        return json.dumps(value, ensure_ascii=False)
    
    return json.dumps([value], ensure_ascii=False)

## 4. Create SQLite Database Table

In [10]:
connection = sqlite3.connect(database_path)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS analyzed_jobs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at TEXT,
    source TEXT,
    job_title TEXT,
    company_name TEXT,
    location TEXT,
    work_mode TEXT,
    employment_type TEXT,
    job_category TEXT,
    seniority_level TEXT,
    minimum_years TEXT,
    required_skills TEXT,
    nice_to_have_skills TEXT,
    programming_languages TEXT,
    frameworks_and_libraries TEXT,
    databases TEXT,
    cloud_and_devops_tools TEXT,
    data_and_ai_tools TEXT,
    testing_tools TEXT,
    other_tools TEXT,
    responsibilities TEXT,
    certifications TEXT,
    language_requirements TEXT,
    soft_skills TEXT,
    metadata TEXT
)
""")

In [11]:
connection.commit()

## 5. Insert Structured Job Posting into SQLite

In [12]:
experience_requirement = structured_job.get("experience_requirement", {}) or {}

job_record = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source": "manual_or_demo_input",
    "job_title": structured_job.get("job_title"),
    "company_name": structured_job.get("company_name"),
    "location": structured_job.get("location"),
    "work_mode": structured_job.get("work_mode"),
    "employment_type": structured_job.get("employment_type"),
    "job_category": structured_job.get("job_category"),
    "seniority_level": experience_requirement.get("seniority_level"),
    "minimum_years": experience_requirement.get("minimum_years"),
    "required_skills": list_to_json_string(structured_job.get("required_skills", [])),
    "nice_to_have_skills": list_to_json_string(structured_job.get("nice_to_have_skills", [])),
    "programming_languages": list_to_json_string(structured_job.get("programming_languages", [])),
    "frameworks_and_libraries": list_to_json_string(structured_job.get("frameworks_and_libraries", [])),
    "databases": list_to_json_string(structured_job.get("databases", [])),
    "cloud_and_devops_tools": list_to_json_string(structured_job.get("cloud_and_devops_tools", [])),
    "data_and_ai_tools": list_to_json_string(structured_job.get("data_and_ai_tools", [])),
    "testing_tools": list_to_json_string(structured_job.get("testing_tools", [])),
    "other_tools": list_to_json_string(structured_job.get("other_tools", [])),
    "responsibilities": list_to_json_string(structured_job.get("responsibilities", [])),
    "certifications": list_to_json_string(structured_job.get("certifications", [])),
    "language_requirements": list_to_json_string(structured_job.get("language_requirements", [])),
    "soft_skills": list_to_json_string(structured_job.get("soft_skills", [])),
    "metadata": json.dumps(structured_job.get("metadata", {}), ensure_ascii=False)
}

In [13]:
cursor.execute("""
INSERT INTO analyzed_jobs (
    created_at,
    source,
    job_title,
    company_name,
    location,
    work_mode,
    employment_type,
    job_category,
    seniority_level,
    minimum_years,
    required_skills,
    nice_to_have_skills,
    programming_languages,
    frameworks_and_libraries,
    databases,
    cloud_and_devops_tools,
    data_and_ai_tools,
    testing_tools,
    other_tools,
    responsibilities,
    certifications,
    language_requirements,
    soft_skills,
    metadata
)
VALUES (
    :created_at,
    :source,
    :job_title,
    :company_name,
    :location,
    :work_mode,
    :employment_type,
    :job_category,
    :seniority_level,
    :minimum_years,
    :required_skills,
    :nice_to_have_skills,
    :programming_languages,
    :frameworks_and_libraries,
    :databases,
    :cloud_and_devops_tools,
    :data_and_ai_tools,
    :testing_tools,
    :other_tools,
    :responsibilities,
    :certifications,
    :language_requirements,
    :soft_skills,
    :metadata
)
""", job_record)

In [14]:
connection.commit()

## 6. Read Stored Jobs from SQLite

In [15]:
df_stored_jobs = pd.read_sql_query(
    "SELECT * FROM analyzed_jobs ORDER BY id DESC",
    connection
)

In [16]:
df_stored_jobs.head()

,id,created_at,source,job_title,company_name,location,work_mode,employment_type,job_category,seniority_level,...,databases,cloud_and_devops_tools,data_and_ai_tools,testing_tools,other_tools,responsibilities,certifications,language_requirements,soft_skills,metadata
0,3,2026-07-12T23:42:08,manual_or_demo_input,Freelance Full-Stack Software Engineer,SkillFit,Remote,Remote,Freelance,Software Development,Mid,...,"[""PostgreSQL"", ""MongoDB""]","[""AWS"", ""Azure"", ""Google Cloud Platform""]",[],[],[],"[""Collaborate with the product team to underst...",[],[],"[""Excellent problem-solving skills"", ""Attentio...","{""source_file"": ""cleaned_it_job_postings.csv"",..."


In [17]:
# last_id = df_stored_jobs["id"].max()
# last_id = 2
# cursor.execute(
#     "DELETE FROM analyzed_jobs WHERE id = ?",
#     (int(last_id),)
# )

# connection.commit()

In [18]:
connection.commit()

In [19]:
df_stored_jobs.head()

,id,created_at,source,job_title,company_name,location,work_mode,employment_type,job_category,seniority_level,...,databases,cloud_and_devops_tools,data_and_ai_tools,testing_tools,other_tools,responsibilities,certifications,language_requirements,soft_skills,metadata
0,3,2026-07-12T23:42:08,manual_or_demo_input,Freelance Full-Stack Software Engineer,SkillFit,Remote,Remote,Freelance,Software Development,Mid,...,"[""PostgreSQL"", ""MongoDB""]","[""AWS"", ""Azure"", ""Google Cloud Platform""]",[],[],[],"[""Collaborate with the product team to underst...",[],[],"[""Excellent problem-solving skills"", ""Attentio...","{""source_file"": ""cleaned_it_job_postings.csv"",..."


## 7. Prepare Text for Skill Statistics

In [20]:
title_col = "clean_job_title"
description_col = "clean_job_description"

df_jobs["market_text"] = (
    df_jobs[title_col].fillna("").astype(str) + " " +
    df_jobs[description_col].fillna("").astype(str)
).str.lower()

In [21]:
df_jobs[["market_text"]].head()

,market_text
0,software engineer job description:goyt is seek...
1,full stack engineer location: remote company o...
2,salesforce vlocity developer role: salesforce ...
3,data architect request: data architectlocation...
4,senior developer this individual will work wit...


## 8. Define IT Skill and Technology Keyword Groups

In [22]:
programming_languages = [
    "python", "java", "javascript", "typescript", "c#", "c++",
    "php", "ruby", "go", "golang", "kotlin", "swift", "scala", "r programming"
]

frameworks_and_libraries = [
    "react", "angular", "vue", "node.js", "node", "express",
    "django", "flask", "fastapi", "spring boot", "spring",
    ".net", "asp.net", "laravel", "symfony", "next.js"
]

databases = [
    "sql", "mysql", "postgresql", "mongodb", "oracle",
    "sqlite", "redis", "elasticsearch", "cassandra", "dynamodb"
]

cloud_and_devops_tools = [
    "aws", "azure", "google cloud", "gcp", "docker", "kubernetes",
    "jenkins", "gitlab", "github actions", "ci/cd", "terraform",
    "ansible", "linux", "nginx"
]

data_ai_bi_tools = [
    "power bi", "tableau", "excel", "pandas", "numpy",
    "spark", "pyspark", "tensorflow", "pytorch", "scikit-learn",
    "machine learning", "deep learning", "data visualization"
]

testing_tools = [
    "selenium", "cypress", "pytest", "junit", "testng",
    "postman", "jest", "unit testing", "automation testing"
]

## 9. Count Keyword Occurrences

In [23]:
def count_keyword_occurrences(df, text_column, keywords):
    results = []

    for keyword in keywords:
        keyword_lower = keyword.lower()

        count = df[text_column].str.contains(
            re.escape(keyword_lower),
            case=False,
            na=False
        ).sum()

        percentage = round((count / len(df)) * 100, 2) if len(df) > 0 else 0

        results.append(
            {
                "keyword": keyword,
                "posting_count": int(count),
                "posting_percentage": percentage
            }
        )

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(by="posting_count", ascending=False).reset_index(drop=True)

    return result_df

## 10. Most Common 

In [24]:
programming_language_stats = count_keyword_occurrences(
    df=df_jobs,
    text_column="market_text",
    keywords=programming_languages
)

In [25]:
programming_language_stats.head(20)

,keyword,posting_count,posting_percentage
0,go,6222,73.77
1,python,3406,40.38
2,java,3150,37.35
3,scala,2485,29.46
4,javascript,1602,18.99
5,c#,907,10.75
6,c++,819,9.71
7,typescript,507,6.01
8,ruby,245,2.90
9,r programming,223,2.64


In [26]:
framework_stats = count_keyword_occurrences(
    df=df_jobs,
    text_column="market_text",
    keywords=frameworks_and_libraries
)

In [27]:
framework_stats.head(20)

,keyword,posting_count,posting_percentage
0,express,1446,17.14
1,react,1086,12.88
2,.net,865,10.26
3,angular,721,8.55
4,node,673,7.98
5,spring,652,7.73
6,node.js,362,4.29
7,spring boot,348,4.13
8,vue,223,2.64
9,asp.net,188,2.23


In [28]:
database_stats = count_keyword_occurrences(
    df=df_jobs,
    text_column="market_text",
    keywords=databases
)

In [29]:
database_stats.head(20)

,keyword,posting_count,posting_percentage
0,sql,3743,44.38
1,oracle,933,11.06
2,mysql,440,5.22
3,postgresql,323,3.83
4,mongodb,309,3.66
5,redis,177,2.10
6,dynamodb,160,1.90
7,cassandra,115,1.36
8,elasticsearch,103,1.22
9,sqlite,25,0.30


In [30]:
cloud_devops_stats = count_keyword_occurrences(
    df=df_jobs,
    text_column="market_text",
    keywords=cloud_and_devops_tools
)

In [31]:
cloud_devops_stats.head(20)

,keyword,posting_count,posting_percentage
0,aws,3380,40.08
1,azure,2285,27.09
2,ci/cd,1269,15.05
3,linux,1238,14.68
4,kubernetes,1128,13.37
5,docker,1005,11.92
6,jenkins,744,8.82
7,gcp,712,8.44
8,terraform,641,7.60
9,google cloud,472,5.60


In [32]:
testing_tool_stats = count_keyword_occurrences(
    df=df_jobs,
    text_column="market_text",
    keywords=testing_tools
)

In [33]:
testing_tool_stats.head(20)

,keyword,posting_count,posting_percentage
0,unit testing,351,4.16
1,selenium,186,2.21
2,junit,129,1.53
3,postman,122,1.45
4,jest,92,1.09
5,cypress,84,1.00
6,automation testing,51,0.60
7,testng,33,0.39
8,pytest,23,0.27


In [50]:
data_ai_bi_stats = count_keyword_occurrences(
    df=df_jobs,
    text_column="market_text",
    keywords=data_ai_bi_tools
)

In [51]:
data_ai_bi_stats.head(20)

,keyword,posting_count,posting_percentage
0,excel,4004,47.47
1,machine learning,1036,12.28
2,tableau,736,8.73
3,spark,630,7.47
4,data visualization,577,6.84
5,power bi,565,6.70
6,pytorch,216,2.56
7,deep learning,214,2.54
8,tensorflow,209,2.48
9,pyspark,140,1.66


## 11. IT Job Category Statistics

In [34]:
if "it_job_category" in df_jobs.columns:
    job_category_stats = (
        df_jobs["it_job_category"]
        .fillna("Unknown")
        .value_counts()
        .reset_index()
    )

    job_category_stats.columns = ["it_job_category", "posting_count"]
    job_category_stats["posting_percentage"] = round(
        job_category_stats["posting_count"] / len(df_jobs) * 100, 2
    )

    job_category_stats
else:
    print("Column 'it_job_category' was not found.")

## 12. Seniority Level Statistics

In [35]:
experience_column_candidates = [
    "experience_level",
    "formatted_experience_level"
]

experience_column = None

for col in experience_column_candidates:
    if col in df_jobs.columns:
        experience_column = col
        break


In [36]:
experience_column

'formatted_experience_level'

In [37]:
if experience_column is not None:
    seniority_stats = (
        df_jobs[experience_column]
        .fillna("Unknown")
        .value_counts()
        .reset_index()
    )

    seniority_stats.columns = ["seniority_level", "posting_count"]
    seniority_stats["posting_percentage"] = round(
        seniority_stats["posting_count"] / len(df_jobs) * 100, 2
    )

    seniority_stats
else:
    print("No seniority column was found.")

## 13. Work Mode Statistics

In [38]:
work_mode_column_candidates = [
    "formatted_work_type",
    "work_type"
]

work_mode_column = None

for col in work_mode_column_candidates:
    if col in df_jobs.columns:
        work_mode_column = col
        break

In [39]:
work_mode_column

'formatted_work_type'

In [40]:
if work_mode_column is not None:
    work_mode_stats = (
        df_jobs[work_mode_column]
        .fillna("Unknown")
        .value_counts()
        .reset_index()
    )

    work_mode_stats.columns = ["work_mode", "posting_count"]
    work_mode_stats["posting_percentage"] = round(
        work_mode_stats["posting_count"] / len(df_jobs) * 100, 2
    )

    work_mode_stats
else:
    print("No work mode column was found.")

## 14. Basic Trend Analysis Over Time

In [41]:
date_column_candidates = [
    "listed_time",
    "original_listed_time"
]

date_column = None

for col in date_column_candidates:
    if col in df_jobs.columns:
        date_column = col
        break

In [42]:
date_column

'listed_time'

In [43]:
def parse_linkedin_datetime(series):
    numeric_series = pd.to_numeric(series, errors="coerce")

    if numeric_series.notna().sum() > 0:
        parsed_dates = pd.to_datetime(numeric_series, unit="ms", errors="coerce")
    else:
        parsed_dates = pd.to_datetime(series, errors="coerce")

    return parsed_dates

In [44]:
if date_column is not None:
    df_jobs["parsed_date"] = parse_linkedin_datetime(df_jobs[date_column])
    df_jobs["year_month"] = df_jobs["parsed_date"].dt.to_period("M").astype(str)

    df_jobs[["parsed_date", "year_month"]].head()
else:
    print("No date column was found.")

In [45]:
if date_column is not None:
    monthly_posting_stats = (
        df_jobs.dropna(subset=["parsed_date"])
        .groupby("year_month")
        .size()
        .reset_index(name="posting_count")
        .sort_values("year_month")
    )

    monthly_posting_stats
else:
    monthly_posting_stats = pd.DataFrame()

## 15 Selected Technology Trends Over Time

In [46]:
selected_trend_keywords = ["python", "sql", "javascript", "react", "docker", "aws"]

if date_column is not None and "year_month" in df_jobs.columns:
    trend_rows = []

    for keyword in selected_trend_keywords:
        keyword_mask = df_jobs["market_text"].str.contains(
            re.escape(keyword),
            case=False,
            na=False
        )

        keyword_monthly = (
            df_jobs[keyword_mask]
            .dropna(subset=["parsed_date"])
            .groupby("year_month")
            .size()
            .reset_index(name="posting_count")
        )

        keyword_monthly["keyword"] = keyword
        trend_rows.append(keyword_monthly)

    if trend_rows:
        technology_trends = pd.concat(trend_rows, ignore_index=True)
    else:
        technology_trends = pd.DataFrame()

    technology_trends.head(20)
else:
    technology_trends = pd.DataFrame()
    print("Technology trend analysis skipped because date information is not available.")

In [47]:
## 16. Save Market Statistics

In [52]:
programming_language_stats.to_csv(
    MARKET_STATS_DIR / "programming_language_stats.csv",
    index=False
)

framework_stats.to_csv(
    MARKET_STATS_DIR / "framework_stats.csv",
    index=False
)

database_stats.to_csv(
    MARKET_STATS_DIR / "database_stats.csv",
    index=False
)

cloud_devops_stats.to_csv(
    MARKET_STATS_DIR / "cloud_devops_stats.csv",
    index=False
)

data_ai_bi_stats.to_csv(
    MARKET_STATS_DIR / "data_ai_bi_stats.csv",
    index=False
)

testing_tool_stats.to_csv(
    MARKET_STATS_DIR / "testing_tool_stats.csv",
    index=False
)

if "job_category_stats" in globals():
    job_category_stats.to_csv(
        MARKET_STATS_DIR / "job_category_stats.csv",
        index=False
    )

if "seniority_stats" in globals():
    seniority_stats.to_csv(
        MARKET_STATS_DIR / "seniority_stats.csv",
        index=False
    )

if "work_mode_stats" in globals():
    work_mode_stats.to_csv(
        MARKET_STATS_DIR / "work_mode_stats.csv",
        index=False
    )

if not monthly_posting_stats.empty:
    monthly_posting_stats.to_csv(
        MARKET_STATS_DIR / "monthly_posting_stats.csv",
        index=False
    )

if not technology_trends.empty:
    technology_trends.to_csv(
        MARKET_STATS_DIR / "technology_trends.csv",
        index=False
    )

print(f"Market statistics saved to: {MARKET_STATS_DIR}")

Market statistics saved to: ..\outputs\market_statistics


## 17. Save Market Statistics Summary

In [53]:
market_summary = {
    "total_cleaned_it_job_postings": int(len(df_jobs)),
    "top_programming_languages": programming_language_stats.head(10).to_dict(orient="records"),
    "top_frameworks": framework_stats.head(10).to_dict(orient="records"),
    "top_databases": database_stats.head(10).to_dict(orient="records"),
    "top_cloud_devops_tools": cloud_devops_stats.head(10).to_dict(orient="records"),
    "top_data_ai_bi_tools": data_ai_bi_stats.head(10).to_dict(orient="records"),
    "top_testing_tools": testing_tool_stats.head(10).to_dict(orient="records")
}

summary_path = MARKET_STATS_DIR / "market_statistics_summary.json"

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(market_summary, file, indent=4, ensure_ascii=False)

print(f"Market statistics summary saved to: {summary_path}")

Market statistics summary saved to: ..\outputs\market_statistics\market_statistics_summary.json


In [54]:
connection.close()

print("SQLite connection closed.")

SQLite connection closed.


In [55]:
market_summary

{'total_cleaned_it_job_postings': 8434,
 'top_programming_languages': [{'keyword': 'go',
   'posting_count': 6222,
   'posting_percentage': 73.77},
  {'keyword': 'python', 'posting_count': 3406, 'posting_percentage': 40.38},
  {'keyword': 'java', 'posting_count': 3150, 'posting_percentage': 37.35},
  {'keyword': 'scala', 'posting_count': 2485, 'posting_percentage': 29.46},
  {'keyword': 'javascript',
   'posting_count': 1602,
   'posting_percentage': 18.99},
  {'keyword': 'c#', 'posting_count': 907, 'posting_percentage': 10.75},
  {'keyword': 'c++', 'posting_count': 819, 'posting_percentage': 9.71},
  {'keyword': 'typescript', 'posting_count': 507, 'posting_percentage': 6.01},
  {'keyword': 'ruby', 'posting_count': 245, 'posting_percentage': 2.9},
  {'keyword': 'r programming',
   'posting_count': 223,
   'posting_percentage': 2.64}],
 'top_frameworks': [{'keyword': 'express',
   'posting_count': 1446,
   'posting_percentage': 17.14},
  {'keyword': 'react', 'posting_count': 1086, 'post